# Four-Module CAT Gadget Circuit

This notebook demonstrates how to generate a Stim circuit that connects four surface-code modules in a 2x2 grid. Adjacent modules are linked horizontally and vertically using a simple CAT gadget.

In [ ]:

import stim


def make_surface_module(circuit, base, d, x_offset=0, y_offset=0):
    """Create a square surface-code patch of distance d."""
    qubits = []
    for r in range(d):
        for c in range(d):
            qid = base + r * d + c
            qubits.append(qid)
            circuit.append(f"QUBIT_COORDS({x_offset + c}, {y_offset + r}) {qid}")
    left = [base + r * d for r in range(d)]
    right = [base + r * d + (d - 1) for r in range(d)]
    top = [base + c for c in range(d)]
    bottom = [base + (d - 1) * d + c for c in range(d)]
    return {
        "data": qubits,
        "left_boundary": left,
        "right_boundary": right,
        "top_boundary": top,
        "bottom_boundary": bottom,
    }


def cat_gadget(circuit, boundary_a, boundary_b):
    """Entangle two boundaries using a simple two-qubit CAT gadget."""
    for qa, qb in zip(boundary_a, boundary_b):
        anc_a = circuit.num_qubits
        anc_b = circuit.num_qubits + 1
        circuit.append('H', anc_a)
        circuit.append('CX', [anc_a, anc_b])
        circuit.append('CX', [anc_a, qa])
        circuit.append('CX', [anc_b, qb])
        circuit.append('M', [anc_a, anc_b])
        circuit.append('TICK')


def build_four_module_grid(d=3, spacing=5):
    """Construct a 2x2 grid of surface-code modules linked by CAT gadgets."""
    circuit = stim.Circuit()
    patches = []
    offsets = [(0, 0), (spacing, 0), (0, spacing), (spacing, spacing)]
    for idx, (ox, oy) in enumerate(offsets):
        base = idx * 1000
        patch = make_surface_module(circuit, base, d, ox, oy)
        patches.append(patch)
    # Horizontal connections
    cat_gadget(circuit, patches[0]['right_boundary'], patches[1]['left_boundary'])
    cat_gadget(circuit, patches[2]['right_boundary'], patches[3]['left_boundary'])
    # Vertical connections
    cat_gadget(circuit, patches[0]['bottom_boundary'], patches[2]['top_boundary'])
    cat_gadget(circuit, patches[1]['bottom_boundary'], patches[3]['top_boundary'])
    return circuit


circuit = build_four_module_grid(d=3)
print(circuit)
